# VAW Phase 1 — Colab run: deep context, Stage A (camera, meters, pose, following)
Needs notebooks 01 (M1/M2) and 02 (M8a tracks) done. CPU is enough. What this does and why: `docs/ARCHITECTURE.md` section 10.
For each clip it measures: is the camera moving, day/night, distances in meters (Hall proxemic zones), who faces whom, looking back, approach from behind,
following with a time lag, contact / reaching (from body keypoints), running away. Results go to `VAW_results/context/`.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt
!python -m pytest -q

## 1. Run Stage A on every clip (resumable: finished clips are skipped)
Prints a table with the share of clips per category showing each behavior. **Compare each category with Normal.** Expect roughly 20-40 minutes
(it reads every video once for the camera-motion estimate). If the session drops, rerun this cell.

In [ ]:
!python -m src.context.features --config configs/colab.yaml --workers 2

## 2. The table again, any time (no recomputation)

In [ ]:
!python -m src.context.features --config configs/colab.yaml --summary

## 3. Sanity check before trusting anything
**Scale**: the median speed of moving people should be about 1-1.5 m/s (normal walking). If it is far off, the assumed field of view
(`context.fov_deg` in the config) is wrong for these cameras.

In [ ]:
import glob, json, numpy as np
from src.context.features import load_context
R = '/content/drive/MyDrive/VAW_results/context'
speeds, moving_cam = [], 0
files = sorted(glob.glob(f'{R}/*/*_scene.json'))
for f in files:
    cat, cid = f.split('/')[-2], f.split('/')[-1].replace('_scene.json', '')
    sc, arr = load_context(R, cat, cid)
    moving_cam += sc['camera']['moving']
    for a in arr.values():
        s = np.r_[a['speed_i'], a['speed_j']]
        s = s[~np.isnan(s)]
        speeds.append(s[s > 0.3])
allspd = np.concatenate([s for s in speeds if len(s)]) if speeds else np.array([])
print(len(files), 'clips;', moving_cam, 'with a moving camera')
print('median moving speed (m/s):', round(float(np.median(allspd)), 2), ' p90:', round(float(np.percentile(allspd, 90)), 2), ' n =', len(allspd))

## 4. Look at examples of each behavior
Prints a few clips per behavior with the time range. Open those clips (`VAW_results/processed/<Category>/`) at those times and judge by eye whether the behavior is really
there. Write down right / wrong counts: that is the precision of each new detector.

In [ ]:
import random
random.seed(1)
found = {}
for f in files:
    cat, cid = f.split('/')[-2], f.split('/')[-1].replace('_scene.json', '')
    sc = json.load(open(f))
    for pair, ev in sc['pairs'].items():
        for name, v in ev.items():
            if isinstance(v, dict) and 'runs' in v:
                beh = name.replace('_i_j', '').replace('_j_i', '').replace('_i', '').replace('_j', '')
                found.setdefault(beh, []).append((cat, cid, pair, name, v['runs'][0]))
for beh, lst in sorted(found.items()):
    print(f"\n{beh}: {len(lst)} events in {len({(c, i) for c, i, *_ in lst})} clips")
    for cat, cid, pair, name, run in random.sample(lst, min(5, len(lst))):
        print(f"   {cat}/{cid}  pair {pair}  {name}  {run[0]}-{run[1]} s")